按顺序运行，默认只显示最终答案。将首个代码单元格的 `VERBOSE` 改为 `True` 可查看学习日志。
模型配置读取根目录 `.env`；提交前清空输出，实际结果记录到 `docs/progress/`。


In [ ]:
from my_agent.output import debug, quiet_stream, run_quietly, set_verbose, with_style, RESPONSE_STYLE

VERBOSE = False  # 排错或学习内部过程时改成 True
set_verbose(VERBOSE)

from my_agent.config import PROJECT_ROOT, load_project_env

env_path = load_project_env()
debug("配置文件：", env_path)

import os
from my_agent.config import DATA_DIR
DATA_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(DATA_DIR)  # 框架相对路径产物集中保存在 data/


In [ ]:
import re
from typing import Optional, List, Tuple
from hello_agents import ReActAgent, HelloAgentsLLM, Config, Message, ToolRegistry
from my_agent.config import load_project_env

In [ ]:
load_project_env()

In [ ]:
MY_REACT_PROMPT = """你是一个具备推理和行动能力的AI助手。你可以通过思考分析问题，然后调用合适的工具来获取信息，最终给出准确的答案。

## 可用工具
{tools}

## 工作流程
请严格按照以下格式进行回应，每次只能执行一个步骤:

Thought: 分析当前问题，思考需要什么信息或采取什么行动。
Action: 选择一个行动，格式必须是以下之一:
- `{{tool_name}}[{{tool_input}}]` - 调用指定工具
- `Finish[最终答案]` - 当你有足够信息给出最终答案时

## 重要提醒
1. 每次回应必须包含Thought和Action两部分
2. 工具调用的格式必须严格遵循:工具名[参数]
3. 只有当你确信有足够信息回答问题时，才使用Finish
4. 如果工具返回的信息不够，继续使用其他工具或相同工具的不同参数

## 当前任务
**Question:** {question}

## 执行历史
{history}

现在开始你的推理和行动:
"""
MY_REACT_PROMPT += "\nFinish 内的最终答案要求：\n" + RESPONSE_STYLE


In [ ]:
class MyReActAgent(ReActAgent):
    """
    重写的ReAct Agent - 推理与行动结合的智能体
    """

    def __init__(
        self,
        name: str,
        llm: HelloAgentsLLM,
        tool_registry: ToolRegistry,
        system_prompt: Optional[str] = None,
        config: Optional[Config] = None,
        max_steps: int = 10,
        custom_prompt: Optional[str] = None
    ):
        super().__init__(name=name, llm=llm, tool_registry=tool_registry,system_prompt=with_style(system_prompt or '你是一个有用的助手。'), config=config,
                        max_steps=max_steps)
        self.current_history: List[str] = []
        self.prompt_template = custom_prompt if custom_prompt else MY_REACT_PROMPT
        debug(f"✅ {name} 初始化完成，最大步数: {max_steps}")

    def run(self,input_text: str, **kwargs):
        self.current_history = []
        current_step = 0
        debug(f"\n🤖 {self.name} 开始处理问题: {input_text}")
        
        while current_step < self.max_steps:
            current_step += 1
            #1、构建提示词
            tools = self.tool_registry.get_tools_description()
            history = "\n".join(self.current_history)
            prompt = self.prompt_template.format(tools = tools, question = input_text, history = history)
            messages = [{"role": "user", "content": prompt}]

            #2、调用LLM
            respond= self.llm.invoke(messages, **kwargs)   

            # 3. 解析输出
            thought,action = self._parse_output(respond)         
            try:
                tool_name, tool_input = self._parse_action(action)
            except ValueError as e:
                debug(f"Action解析失败: {e}")
            
                self.current_history.append(
                    f"模型上次输出：{respond}\n"
                    "Observation: Action格式错误。"
                    "请只输出一个完整的 Tool[参数] 或 Finish[答案]。"
                )
                continue

            # 4、检测完成条件
            if tool_name =="Finish":
                final_answer = tool_input
                self.add_message(Message(input_text, "user"))
                self.add_message(Message(final_answer, "assistant"))
                return final_answer

            #5、调用工具
            try:
                observation = self.tool_registry.execute_tool(tool_name,tool_input)
            except Exception as e:
                observation = f"工具调用失败{e}"
            debug("Observation:",observation)

            #6、更新历史记录
            self.current_history.append(f"Action: {action}")
            self.current_history.append(f"Observation: {observation}")

        # 达到最大步数
        final_answer = "抱歉，我无法在限定步数内完成这个任务。"
        self.add_message(Message(input_text, "user"))
        self.add_message(Message(final_answer, "assistant"))
        return final_answer  
            
            
        
        
    def _parse_output(self, response_text: str):
        """从LLM输出中提取Thought和Action"""
    
        thought_match = re.search(
            r"(?m)^[ \t]*Thought\s*:\s*(.*)$",
            response_text
        )
    
        action_match = re.search(
            r"(?ms)^[ \t]*Action\s*:\s*(.*)",
            response_text
        )
    
        thought = thought_match.group(1).strip() if thought_match else ""
        action = action_match.group(1).strip() if action_match else ""
    
        return thought, action


    def _parse_action(self, action: str):
        """把 calculator[15 * 8 + 32] 解析为工具名和参数"""
    
        match = re.fullmatch(
            r"([A-Za-z_]\w*)\[(.*)\]",
            action.strip(),
            flags=re.S
        )
    
        if not match:
            raise ValueError(f"无法解析Action: {action}")
    
        tool_name = match.group(1)
        tool_input = match.group(2)
    
        return tool_name, tool_input

In [ ]:
from hello_agents.tools import CalculatorTool

llm = HelloAgentsLLM()

registry = ToolRegistry()
run_quietly(registry.register_tool, CalculatorTool())

debug(registry.get_tools_description())

agent = MyReActAgent(
    name="我的ReAct助手",
    llm=llm,
    tool_registry=registry,
    max_steps=5
)

result = run_quietly(agent.run, "假设某商品原价250元，先打八折，然后加收折后价格6%的服务费，最后使用一张30元优惠券。请分别计算每个步骤的金额以及最终价格，每一步必须单独调用一次计算器。")
print(result)

In [ ]:
debug(registry.list_tools())
debug(agent.tool_registry.get_tools_description())